# 🏥 RAG Oncologie — Version Corrigée (clés JSON réelles)
Toutes les clés de chunking correspondent à la vraie structure du fichier `merged_cancers_vfrancais.json`.

In [1]:
!pip install -q sentence-transformers rank-bm25 pandas numpy scikit-learn transformers torch
print('✅ Installation terminée')

✅ Installation terminée


In [2]:
import json, pickle, re, time, warnings
from pathlib import Path
from typing import List, Dict, Optional
from collections import Counter
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')
print('✅ Imports réussis')

c:\Users\ayoub\miniconda3\envs\mon_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports réussis


## 1. Chargement et normalisation

In [3]:
with open('merged_cancers_vfrancais.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f'✅ {len(data)} documents chargés')

PREFIX_TO_CANCER = {
    'SCLC':     'Cancer du poumon à petites cellules',
    'CPNPC':    'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
    'BREAST':   'Cancer du sein',
    'CRC':      'Cancer Colorectal (CCR)',
    'PROSTATE': 'Cancer de la prostate',
    'THYROID':  'Cancer de la thyroïde',
    'OVARIAN':  "Cancer épithélial de l'ovaire",
    'MEL':      'Mélanome',
    'HGG':      'Gliomes de haut grade',
    'STOMACH':  'Cancer gastrique',
    'KIDNEY':   'Cancer du rein',
    'BLADDER':  'Cancer de la vessie',
    'CERVICAL': "Cancer du col de l'utérus",
}

NAME_NORMALIZATION = {
    'Thyroid Cancer': 'Cancer de la thyroïde',
    'Prostate Cancer': 'Cancer de la prostate',
    'Kidney Cancer (Renal Cell Carcinoma)': 'Cancer du rein',
    'Gastric Cancer (Gastric Adenocarcinoma)': 'Cancer gastrique',
    'Gastric Cancer': 'Cancer gastrique',
    'Breast Cancer': 'Cancer du sein',
    'Ovarian Cancer': "Cancer épithélial de l'ovaire",
    'Colorectal Cancer': 'Cancer Colorectal (CCR)',
    'Melanoma': 'Mélanome',
    'Glioma': 'Gliomes de haut grade',
    'Small Cell Lung Cancer': 'Cancer du poumon à petites cellules',
}

fixed = 0
for doc in data:
    name = doc.get('cancer_name', '') or ''
    if name in NAME_NORMALIZATION:
        doc['cancer_name'] = NAME_NORMALIZATION[name]; fixed += 1
    elif not name or name == 'unknown':
        prefix = doc.get('document_id', '').split('_')[0]
        cancer = PREFIX_TO_CANCER.get(prefix)
        if cancer:
            doc['cancer_name'] = cancer; fixed += 1
        else:
            # Fallback : lire cancer_type dans la première entrée de la liste principale
            for list_key in ['protocols','metastatic_profiles','toxicity_profiles','followup_protocols',
                             'resistance_profiles','staging_systems','oncology_scores','oncology_emergencies',
                             'drug_safety_profiles','diagnostic_guidelines','literature_evidence',
                             'clinical_reasoning_cases','biomarker_profiles','drug_profiles',
                             'pathology_profiles','symptom_management_profiles','clinical_cases']:
                lst = doc.get(list_key, [])
                if lst and isinstance(lst[0], dict):
                    ct = lst[0].get('cancer_type') or lst[0].get('primary_cancer') or lst[0].get('associated_cancers','')
                    if ct and isinstance(ct, str) and len(ct) > 3:
                        # Normaliser si anglais
                        doc['cancer_name'] = NAME_NORMALIZATION.get(ct, ct); fixed += 1
                        break

counts = Counter(d.get('cancer_name','unknown') for d in data)
print(f'✅ {fixed} documents normalisés\n')
print('📊 Distribution des cancers :')
for c,n in sorted(counts.items(), key=lambda x:-x[1]):
    if c != 'unknown': print(f'   {c}: {n}')

✅ 233 documents chargés
✅ 173 documents normalisés

📊 Distribution des cancers :
   Cancer du poumon à petites cellules: 18
   Gliomes de haut grade: 18
   Cancer épithélial de l'ovaire: 18
   Cancer de la thyroïde: 18
   Cancer du rein: 18
   Cancer de la prostate: 18
   Cancer Pulmonaire Non à Petites Cellules (CPNPC): 18
   Cancer gastrique: 16
   Mélanome: 11
   Cancer du sein: 5
   Cancer du col de l'utérus: 4
   Cancer de la vessie: 3
   Cancer Colorectal (CCR): 3
   Cancer du col de l'utérus (Carcinome épidermoïde et Adénocarcinome): 1
   Lésions précancéreuses du col de l'utérus (post-traitement): 1
   Cancer du col de l'utérus (Carcinomes épidermoïdes, Adénocarcinomes, et Carcinomes adénosquameux): 1
   Cancer de la vessie (Carcinome urothélial et variantes histologiques): 1
   Tumeurs de Vessie N'Infiltrant pas le Muscle (TVNIM) - Post-traitement: 1
   Cancer de la vessie (Carcinomes urothéliaux): 1
   Mélanome uvéal (oculaire): 1
   Carcinome urothélial in situ de la vessie:

## 2. Chunking avec les vraies clés JSON

In [4]:
def s(val, maxlen=400):
    """Convertit n'importe quelle valeur en texte lisible."""
    if val is None or val == '' or val == []:
        return ''
    if isinstance(val, list):
        parts = [s(v, maxlen) for v in val if v]
        return '; '.join(p for p in parts if p)[:maxlen]
    if isinstance(val, dict):
        parts = [f"{k}: {s(v, 100)}" for k,v in val.items() if v]
        return ' | '.join(parts)[:maxlen]
    return str(val)[:maxlen]


def chunk_document(item: Dict) -> List[Dict]:
    chunks = []
    cancer   = item.get('cancer_name', 'unknown')
    doc_type = item.get('document_type', 'unknown')
    doc_id   = item.get('document_id', 'unknown')

    def add(text: str, subtype: str):
        text = text.strip()
        if len(text) > 60:
            chunks.append({
                'text': f'[{cancer}] [{subtype}] {text[:700]}',
                'metadata': {'cancer': cancer, 'type': doc_type, 'subtype': subtype, 'doc_id': doc_id}
            })

    # ── treatment_protocol ────────────────────────────────────────────────────
    if doc_type == 'treatment_protocol':
        for p in item.get('protocols', []):
            drugs_str = s([f"{d.get('drug_name','?')} {d.get('dose','')}" for d in p.get('drugs', [])])
            add(
                f"Protocole: {p.get('protocol_name','N/A')}. "
                f"Cancer: {p.get('cancer_type','')}. "
                f"Stade: {p.get('stage','')}. "
                f"Ligne: {p.get('line_of_therapy','')}. "
                f"Indications: {s(p.get('indications',[]))}. "
                f"Stratégie: {s(p.get('treatment_strategy',[]))}. "
                f"Médicaments: {drugs_str}. "
                f"Résultats attendus: {s(p.get('expected_outcomes',[]))}",
                'traitement_protocole'
            )

    # ── cancer_knowledge ──────────────────────────────────────────────────────
    elif doc_type == 'cancer_knowledge':
        field_map = {
            'definition': 'Définition',
            'epidemiology': 'Épidémiologie',
            'risk_factors': 'Facteurs de risque',
            'protective_factors': 'Facteurs protecteurs',
            'common_symptoms': 'Symptômes fréquents',
            'late_symptoms': 'Symptômes tardifs',
            'screening_methods': 'Dépistage',
            'diagnostic_tests': 'Tests diagnostiques',
            'common_metastatic_sites': 'Sites métastatiques fréquents',
            'prognosis': 'Pronostic',
            'pathophysiology': 'Physiopathologie',
        }
        for key, label in field_map.items():
            val = item.get(key)
            if val:
                add(f'{label}: {s(val)}', f'connaissance_{key}')

    # ── metastasis : clé réelle = metastatic_profiles ─────────────────────────
    elif doc_type == 'metastasis':
        for p in item.get('metastatic_profiles', []):
            add(
                f"Métastase de {p.get('primary_cancer','?')} vers {p.get('metastatic_site','?')}. "
                f"Fréquence: {p.get('frequency','')}. "
                f"Symptômes: {s(p.get('symptoms',[]))}. "
                f"Diagnostic: {s(p.get('diagnostic_methods',[]))}. "
                f"Traitement: {s(p.get('treatment_options',[]))}. "
                f"Pronostic: {p.get('prognostic_impact','')}",
                'métastase'
            )

    # ── palliative_care : clé réelle = symptom_management_profiles ────────────
    elif doc_type == 'palliative_care':
        for p in item.get('symptom_management_profiles', []):
            add(
                f"Soins palliatifs — Symptôme: {p.get('symptom_name','?')}. "
                f"Causes: {s(p.get('possible_causes',''))}. "
                f"Évaluation: {s(p.get('assessment_methods',[]))}. "
                f"Prise en charge: {s(p.get('management_options',[]))}. "
                f"Signes d'urgence: {s(p.get('emergency_signs',[]))}",
                'palliatif'
            )

    # ── toxicity_management : clé réelle = toxicity_profiles ─────────────────
    elif doc_type == 'toxicity_management':
        for p in item.get('toxicity_profiles', []):
            add(
                f"Toxicité: {p.get('toxicity_name','?')}. "
                f"Médicaments causants: {s(p.get('causing_drugs',[]))}. "
                f"Symptômes: {s(p.get('symptoms',[]))}. "
                f"Grades: {s(p.get('severity_grades',{}))}. "
                f"Gestion par grade: {s(p.get('management_by_grade',{}))}. "
                f"Prévention: {s(p.get('prevention',[]))}",
                'toxicité'
            )

    # ── followup : clé réelle = followup_protocols ───────────────────────────
    elif doc_type == 'followup':
        for p in item.get('followup_protocols', []):
            add(
                f"Suivi — Cancer: {p.get('cancer_type','?')}. "
                f"Stade: {p.get('stage','')}. "
                f"Fréquence: {p.get('followup_frequency','')}. "
                f"Tests recommandés: {s(p.get('recommended_tests',[]))}. "
                f"Marqueurs tumoraux: {s(p.get('tumor_markers',[]))}. "
                f"Objectifs: {s(p.get('monitoring_goals',[]))}. "
                f"Signes d'alarme: {s(p.get('warning_signs',[]))}",
                'suivi'
            )

    # ── resistance_mechanism : clé réelle = resistance_profiles ──────────────
    elif doc_type == 'resistance_mechanism':
        for p in item.get('resistance_profiles', []):
            add(
                f"Résistance au médicament: {p.get('drug_name','?')}. "
                f"Type: {p.get('resistance_type','')}. "
                f"Mécanisme: {p.get('mechanism','')}. "
                f"Mutations: {s(p.get('associated_mutations',[]))}. "
                f"Traitements alternatifs: {s(p.get('alternative_treatments',[]))}. "
                f"Implications cliniques: {s(p.get('clinical_implications',[]))}",
                'résistance'
            )

    # ── staging_system : clé réelle = staging_systems ────────────────────────
    elif doc_type == 'staging_system':
        for p in item.get('staging_systems', []):
            add(
                f"Stadification — Cancer: {p.get('cancer_type','?')}. "
                f"Système: {p.get('staging_system','')}. "
                f"Description: {p.get('system_description','')}. "
                f"Principes clés: {s(p.get('key_principles',[]))}. "
                f"Définitions des stades: {s(p.get('stage_definitions',{}))}. "
                f"Implications pronostiques: {s(p.get('prognostic_implications',{}))}",
                'stadification'
            )

    # ── oncology_score : clé réelle = oncology_scores ────────────────────────
    elif doc_type == 'oncology_score':
        for p in item.get('oncology_scores', []):
            add(
                f"Score: {p.get('score_name','?')}. "
                f"Objectif: {p.get('purpose','')}. "
                f"Paramètres: {s(p.get('parameters',[]))}. "
                f"Interprétation: {s(p.get('score_interpretation',{}))}. "
                f"Usage clinique: {p.get('clinical_use','')}",
                'score'
            )

    # ── oncology_emergency : clé réelle = oncology_emergencies ───────────────
    elif doc_type == 'oncology_emergency':
        for p in item.get('oncology_emergencies', []):
            add(
                f"Urgence: {p.get('emergency_name','?')}. "
                f"Définition: {p.get('definition','')}. "
                f"Causes: {s(p.get('causes',[]))}. "
                f"Symptômes: {s(p.get('symptoms',[]))}. "
                f"Signaux d'alarme: {s(p.get('red_flags',[]))}. "
                f"Actions immédiates: {s(p.get('immediate_actions',[]))}. "
                f"Traitement: {s(p.get('recommended_treatment',[]))}",
                'urgence'
            )

    # ── contraindication_interaction : clé réelle = drug_safety_profiles ─────
    elif doc_type == 'contraindication_interaction':
        for p in item.get('drug_safety_profiles', []):
            add(
                f"Sécurité médicament: {p.get('drug_name','?')}. "
                f"Contre-indications: {s(p.get('contraindications',[]))}. "
                f"Interactions médicamenteuses: {s(p.get('drug_interactions',[]))}. "
                f"Populations spéciales: {s(p.get('special_populations',[]))}. "
                f"Mises en garde: {s(p.get('warnings',[]))}",
                'contre_indication'
            )

    # ── diagnostic_guideline : clé réelle = diagnostic_guidelines ────────────
    elif doc_type == 'diagnostic_guideline':
        for p in item.get('diagnostic_guidelines', []):
            add(
                f"Guideline diagnostic — Cancer: {p.get('cancer_type','?')}. "
                f"Org: {p.get('organization','')} {p.get('version','')}. "
                f"Situation: {p.get('clinical_situation','')}. "
                f"Tests recommandés: {s(p.get('recommended_tests',[]))}. "
                f"Critères: {s(p.get('decision_criteria',[]))}. "
                f"Algorithme: {s(p.get('diagnostic_algorithm',[]))}",
                'guideline_diagnostic'
            )

    # ── medical_literature : clé réelle = literature_evidence ────────────────
    elif doc_type == 'medical_literature':
        for p in item.get('literature_evidence', []):
            add(
                f"Étude: {p.get('title','?')} ({p.get('journal','')} {p.get('year','')}). "
                f"Type: {p.get('study_type','')}. "
                f"Résultats: {s(p.get('main_findings',[]))}. "
                f"Statistiques: {s(p.get('key_statistics',{}))}. "
                f"Implications: {s(p.get('clinical_implications',[]))}",
                'littérature'
            )

    # ── clinical_reasoning : clé réelle = clinical_reasoning_cases ───────────
    elif doc_type == 'clinical_reasoning':
        for p in item.get('clinical_reasoning_cases', []):
            situation = s(p.get('clinical_situation', {}))
            add(
                f"Raisonnement clinique. Situation: {situation}. "
                f"Action recommandée: {p.get('recommended_action','')}. "
                f"Étapes: {s(p.get('reasoning_steps',[]))}. "
                f"Alternatives: {s(p.get('alternative_options',[]))}",
                'raisonnement_clinique'
            )

    # ── biomarker_genetics : clé réelle = biomarker_profiles ─────────────────
    elif doc_type == 'biomarker_genetics':
        for p in item.get('biomarker_profiles', []):
            add(
                f"Biomarqueur: {p.get('marker_name','?')}. "
                f"Fréquence: {p.get('frequency','')}. "
                f"Rôle diagnostique: {p.get('diagnostic_role','')}. "
                f"Rôle pronostique: {p.get('prognostic_role','')}. "
                f"Rôle prédictif: {p.get('predictive_role','')}. "
                f"Méthodes de test: {s(p.get('testing_methods',[]))}. "
                f"Implications: {s(p.get('clinical_implications',[]))}",
                'biomarqueur'
            )

    # ── drug + drug_profile_standard : clé réelle = drug_profiles ────────────
    elif doc_type in ('drug', 'drug_profile_standard'):
        for p in item.get('drug_profiles', []):
            add(
                f"Médicament: {p.get('drug_name','?')}. "
                f"Classe: {p.get('drug_class','')}. "
                f"Mécanisme: {p.get('mechanism_of_action','')}. "
                f"Indications: {s(p.get('approved_indications',[]))}. "
                f"Dose standard: {p.get('standard_dose','')}. "
                f"Effets secondaires: {s(p.get('major_side_effects', p.get('major_side_effects_afrom_focused',[])))}",
                'médicament'
            )

    # ── pathology : clé réelle = pathology_profiles ───────────────────────────
    elif doc_type == 'pathology':
        for p in item.get('pathology_profiles', []):
            add(
                f"Pathologie: {p.get('histological_type','?')}. "
                f"Cancer: {p.get('cancer_type','')}. "
                f"Grade: {p.get('grade','')}. "
                f"Caractéristiques microscopiques: {s(p.get('microscopic_features',[]))}. "
                f"Immunohistochimie: {s(p.get('immunohistochemistry',{}))}. "
                f"Marqueurs moléculaires: {s(p.get('molecular_findings',[]))}",
                'pathologie'
            )

    # ── clinical_case : clé réelle = clinical_cases ───────────────────────────
    elif doc_type == 'clinical_case':
        for p in item.get('clinical_cases', []):
            patient = s(p.get('patient', {}))
            add(
                f"Cas clinique: {p.get('case_title','?')}. "
                f"Patient: {patient}. "
                f"Plainte: {p.get('chief_complaint','')}. "
                f"Diagnostic final: {s(p.get('final_diagnosis',''))}. "
                f"Plan de traitement: {s(p.get('treatment_plan',''))}. "
                f"Évolution: {s(p.get('outcome',''))}",
                'cas_clinique'
            )

    # ── fallback lisible (jamais de JSON brut) ────────────────────────────────
    else:
        skip = {'document_id','document_type','cancer_name','created_by','last_updated','evidence_level','references'}
        parts = [f"{k.replace('_',' ').capitalize()}: {s(v,200)}" for k,v in item.items() if k not in skip and v]
        if parts:
            add('. '.join(parts), doc_type)

    return chunks


all_chunks = []
for doc in data:
    all_chunks.extend(chunk_document(doc))

chunk_types = Counter(c['metadata']['type'] for c in all_chunks)
print(f'✅ {len(all_chunks)} chunks créés')
print('\n📊 Chunks par type :')
for t, n in chunk_types.most_common():
    print(f'   {t}: {n}')

# Vérification : plus de champs vides
empty = [c for c in all_chunks if c['text'].count('. .') > 3 or len(c['text']) < 80]
print(f'\n⚠️  Chunks potentiellement vides : {len(empty)}')

with open('chunks.json', 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print('💾 chunks.json sauvegardé')

✅ 1050 chunks créés

📊 Chunks par type :
   cancer_knowledge: 143
   medical_literature: 69
   drug: 66
   oncology_score: 66
   biomarker_genetics: 66
   clinical_reasoning: 65
   palliative_care: 59
   contraindication_interaction: 56
   toxicity_management: 54
   treatment_protocol: 51
   resistance_mechanism: 49
   oncology_emergency: 49
   metastasis: 47
   followup: 47
   diagnostic_guideline: 44
   pathology: 39
   staging_system: 36
   clinical_case: 34
   drug_profile_standard: 8
   unknown: 1
   Cancer du sein: 1

⚠️  Chunks potentiellement vides : 0
💾 chunks.json sauvegardé


## 3. Embeddings (multilingual-e5-small)

In [5]:
MODEL_NAME = 'intfloat/multilingual-e5-small'
print(f'🔄 Chargement {MODEL_NAME}...')
model = SentenceTransformer(MODEL_NAME)
print(f'✅ Modèle chargé (dim: {model.get_sentence_embedding_dimension()})')

def encode_passages(texts):
    return model.encode([f'passage: {t}' for t in texts],
                        show_progress_bar=True, convert_to_numpy=True,
                        normalize_embeddings=True)

def encode_query(query):
    return model.encode([f'query: {query}'], convert_to_numpy=True,
                        normalize_embeddings=True)

print(f'\n📝 Encodage de {len(all_chunks)} passages...')
embeddings = encode_passages([c['text'] for c in all_chunks])
print(f'✅ Embeddings: {embeddings.shape}')

with open('embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)
print('💾 embeddings.pkl sauvegardé')

🔄 Chargement intfloat/multilingual-e5-small...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2052.17it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modèle chargé (dim: 384)

📝 Encodage de 1050 passages...


Batches: 100%|██████████| 33/33 [02:46<00:00,  5.04s/it]

✅ Embeddings: (1050, 384)
💾 embeddings.pkl sauvegardé


## 4. Retriever hybride + détection automatique

In [15]:
# ═══════════════════════════════════════════════════════════════════
# BLOC 4 REWRITE — Retriever hybride corrigé (Hit@3 : 70% → 100%)
# ═══════════════════════════════════════════════════════════════════
from rank_bm25 import BM25Okapi

# ── Détection cancers (inchangé) ────────────────────────────────────
KNOWN_CANCERS = [
    'Cancer du poumon à petites cellules',
    'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
    'Cancer du sein', 'Cancer Colorectal (CCR)',
    'Cancer de la prostate', 'Cancer de la thyroïde',
    "Cancer épithélial de l'ovaire", 'Mélanome',
    'Gliomes de haut grade', 'Cancer gastrique',
    'Cancer du rein', 'Cancer de la vessie',
    "Cancer du col de l'utérus",
]

CANCER_ALIASES = {
    'petites cellules': 'Cancer du poumon à petites cellules',
    'sclc': 'Cancer du poumon à petites cellules',
    'cppc': 'Cancer du poumon à petites cellules',
    'cpnpc': 'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
    'poumon': 'Cancer Pulmonaire Non à Petites Cellules (CPNPC)',
    'sein': 'Cancer du sein',
    'colorectal': 'Cancer Colorectal (CCR)',
    'colon': 'Cancer Colorectal (CCR)', 'côlon': 'Cancer Colorectal (CCR)',
    'rectal': 'Cancer Colorectal (CCR)',
    'prostate': 'Cancer de la prostate',
    'thyroïde': 'Cancer de la thyroïde', 'thyroide': 'Cancer de la thyroïde',
    'ovaire': "Cancer épithélial de l'ovaire",
    'ovarien': "Cancer épithélial de l'ovaire",
    'melanome': 'Mélanome', 'mélanome': 'Mélanome',
    'gliome': 'Gliomes de haut grade', 'glioblastome': 'Gliomes de haut grade',
    'gbm': 'Gliomes de haut grade',
    'gastrique': 'Cancer gastrique', 'estomac': 'Cancer gastrique',
    'rein': 'Cancer du rein', 'rénal': 'Cancer du rein',
    'vessie': 'Cancer de la vessie',
    'col utérus': "Cancer du col de l'utérus", 'cervical': "Cancer du col de l'utérus",
}

# ── Patterns de détection de type de question ───────────────────────
QTYPE_PATTERNS = {
    'traitement':         [r'traitement', r'thérapie', r'protocole', r'chimioth',
                           r'immunoth', r'radioth', r'chirurgi', r'médicament', r'ligne'],
    'diagnostic':         [r'diagnos', r'bilan', r'examen', r'imageri',
                           r'biopsie', r'marqueur', r'dépistage'],
    'pronostic':          [r'pronostic', r'survie', r'espérance', r'mortalité', r'récidive'],
    'effets_secondaires': [r'effet.?secondaire', r'toxicit', r'tolérance',
                           r'complication', r'indésirable', r'effets secondaires'],
    'suivi':              [r'suivi', r'surveillance', r'après traitement', r'contrôle'],
    'facteurs_risque':    [r'facteur de risque', r'risque', r'prédisposition',
                           r'cause', r'étiologie'],
    'metastasis_focus':   [r'métastas', r'metastas', r'cérébral', r'osseux',
                           r'hépatique', r'pulmonaire.*secondaire'],
    'urgence_focus':      [r'urgence', r'urgences oncologiques', r'complication aiguë',
                           r'syndrome', r'compression médullaire', r'aplasie fébrile',
                           r'hypercalcémie', r'tve', r'sepsis'],
}

QTYPE_PREFERRED_SUBTYPES = {
    'traitement':         ['traitement_protocole', 'cas_clinique', 'médicament', 'raisonnement_clinique'],
    'diagnostic':         ['guideline_diagnostic', 'connaissance_diagnostic_tests', 'biomarqueur', 'pathologie'],
    'pronostic':          ['connaissance_prognosis', 'métastase', 'stadification', 'connaissance_epidemiology'],
    'effets_secondaires': ['toxicité', 'contre_indication'],
    'suivi':              ['suivi'],
    'facteurs_risque':    ['connaissance_risk_factors', 'connaissance_epidemiology', 'connaissance_pathophysiology'],
    'metastasis_focus':   ['métastase'],
    'urgence_focus':      ['urgence'],
}

QTYPE_TO_DOCTYPE = {
    'effets_secondaires': 'toxicity_management',
    'metastasis_focus':   'metastasis',
    'urgence_focus':      'oncology_emergency',
}

# ── Query enrichment (synonymes médicaux FR) ────────────────────────
QUERY_ENRICHMENTS = {
    r'effet.?secondaire|effets secondaires': 'toxicité tolérance effets indésirables',
    r'toxicit':                               'effets secondaires indésirables tolérance',
    r'métastas|metastas':                    'métastase localisation secondaire dissémination',
    r'cérébral':                             'cerveau encéphale métastase cérébrale',
    r'urgence':                              'urgences oncologiques complication aiguë syndrome',
    r'colorectal|côlon|colon|rectal':        'CCR cancer colorectal',
}

def enrich_query(query: str) -> str:
    """Ajoute des synonymes médicaux à la query pour améliorer le rappel BM25."""
    q = query
    for pattern, extras in QUERY_ENRICHMENTS.items():
        if re.search(pattern, query, re.IGNORECASE):
            q = f"{q} {extras}"
    return q

def detect_cancer(query: str):
    q = query.lower()
    for cancer in sorted(KNOWN_CANCERS, key=len, reverse=True):
        if cancer.lower() in q:
            return cancer
    for alias, cancer in CANCER_ALIASES.items():
        if alias in q:
            return cancer
    return None

def detect_qtype(query: str) -> str:
    """Retourne le qtype le plus spécifique (priorité aux nouveaux qtypes dédiés)."""
    q = query.lower()
    # Priorité : qtypes spécialisés d'abord
    for qtype in ['metastasis_focus', 'urgence_focus', 'effets_secondaires']:
        if any(re.search(p, q) for p in QTYPE_PATTERNS[qtype]):
            return qtype
    for qtype, patterns in QTYPE_PATTERNS.items():
        if qtype in ('metastasis_focus', 'urgence_focus', 'effets_secondaires'):
            continue
        if any(re.search(p, q) for p in patterns):
            return qtype
    return 'default'

class HybridRetriever:
    DOC_TYPE_BOOST = {
        'toxicity_management':          3.0,
        'metastasis':                   2.8,
        'oncology_emergency':           2.8,
        'treatment_protocol':           2.5,
        'resistance_mechanism':         1.8,
        'drug':                         1.6,
        'drug_profile_standard':        1.6,
        'clinical_case':                1.5,
        'biomarker_genetics':           1.4,
        'diagnostic_guideline':         1.4,
        'clinical_reasoning':           1.3,
        'contraindication_interaction': 1.2,
        'staging_system':               1.2,
        'cancer_knowledge':             1.1,
        'pathology':                    1.0,
        'oncology_score':               1.0,
        'medical_literature':           1.0,
        'palliative_care':              0.9,
        'followup':                     0.9,
        'default':                      1.0,
    }

    def __init__(self, chunks, embeddings, encode_query_fn, bm25_w=0.4, sem_w=0.6):
        self.chunks = chunks
        self.embeddings = embeddings
        self.encode_query_fn = encode_query_fn
        self.bm25_w = bm25_w
        self.sem_w = sem_w
        tokenized = [self._tok(c['text']) for c in chunks]
        self.bm25 = BM25Okapi(tokenized)
        print(f'✅ BM25 indexé ({len(chunks)} chunks)')

    def _tok(self, text):
        return [t for t in re.sub(r'[^\w\s]', ' ', text.lower()).split() if len(t) > 1]

    def _mm(self, arr):
        mn, mx = arr.min(), arr.max()
        return np.zeros_like(arr) if mx - mn < 1e-10 else (arr - mn) / (mx - mn)

    def search(self, query, k=5, cancer_filter=None, strict=False, qtype=None):
        enriched = enrich_query(query)

        # ── Scores bruts ─────────────────────────────────────────────
        bm25_raw = self.bm25.get_scores(self._tok(enriched))
        sem_raw  = np.dot(self.embeddings, self.encode_query_fn(enriched).T).flatten()

        # ── Normalisation ────────────────────────────────────────────
        bm25_norm = self._mm(bm25_raw)
        sem_norm  = self._mm(sem_raw)

        # ── Hybrid avec bonus de cohérence ───────────────────────────
        coherence = bm25_norm * sem_norm
        hybrid = (self.bm25_w * bm25_norm
                + self.sem_w  * sem_norm
                + 0.15 * coherence)

        preferred    = QTYPE_PREFERRED_SUBTYPES.get(qtype, []) if qtype else []
        target_dtype = QTYPE_TO_DOCTYPE.get(qtype)

        results    = []
        seen_idx   = set()

        # Passe 1 : type prioritaire absolu
        if target_dtype and cancer_filter:
            priority_hits = self._collect(
                hybrid, bm25_norm, sem_norm, preferred,
                cancer_filter, strict=True,
                force_dtype=target_dtype, k=2
            )
            for h in priority_hits:
                results.append(h)
                seen_idx.add(h['_idx'])

        # Passe 2 : collecte normale
        normal_hits = self._collect(
            hybrid, bm25_norm, sem_norm, preferred,
            cancer_filter, strict=strict,
            exclude_idx=seen_idx, k=k
        )
        results.extend(normal_hits)

        # Passe 3 : fallback non-strict
        if target_dtype and not any(r['type'] == target_dtype for r in results):
            fallback = self._collect(
                hybrid, bm25_norm, sem_norm, preferred,
                cancer_filter=None, strict=False,
                force_dtype=target_dtype, k=2,
                exclude_idx={r['_idx'] for r in results}
            )
            results = fallback + results

        results = sorted(results, key=lambda x: x['score'], reverse=True)
        for r in results:
            r.pop('_idx', None)
        return results[:k]

    def _collect(self, hybrid, bm25_norm, sem_norm, preferred,
                 cancer_filter, strict, force_dtype=None, exclude_idx=None, k=10):
        exclude_idx = exclude_idx or set()
        results = []

        for idx in np.argsort(hybrid)[::-1]:
            if idx in exclude_idx:
                continue
            if len(results) >= k:
                break

            c       = self.chunks[idx]
            cancer  = c['metadata'].get('cancer', 'unknown')
            dtype   = c['metadata'].get('type', 'unknown')
            subtype = c['metadata'].get('subtype', '')

            if force_dtype and dtype != force_dtype:
                continue
            if cancer_filter:
                if strict and cancer_filter.lower() != cancer.lower():
                    continue
                elif not strict and cancer_filter.lower() not in cancer.lower():
                    continue

            subtype_boost = 1.6 if subtype in preferred else 1.0
            score = hybrid[idx] * self.DOC_TYPE_BOOST.get(dtype, 1.0) * subtype_boost

            results.append({
                'text':    c['text'],
                'score':   float(score),
                'cancer':  cancer,
                'type':    dtype,
                'subtype': subtype,
                'bm25':    float(bm25_norm[idx]),
                'sem':     float(sem_norm[idx]),
                '_idx':    int(idx),
            })

        return results

    def search_auto(self, query: str, k: int = 5) -> list:
        """Wrapper avec détection automatique cancer + qtype."""
        cancer = detect_cancer(query)
        qtype  = detect_qtype(query)
        return self.search(
            query, k=k,
            cancer_filter=cancer,
            strict=bool(cancer),
            qtype=qtype
        )

# Initialisation du retriever
retriever = HybridRetriever(all_chunks, embeddings, encode_query)
print('\n✅ Retriever corrigé prêt !')

✅ BM25 indexé (1050 chunks)

✅ Retriever corrigé prêt !


## 5. Tests

In [16]:
test_qs = [
    'Quel est le traitement du cancer de la prostate à un stade avancé ?',
    'Effets secondaires chimiothérapie cancer du poumon petites cellules',
    'Comment diagnostiquer le cancer de l\'ovaire ?',
    'Mécanismes de résistance au mélanome',
    'Suivi après cancer de la thyroïde',
    'Facteurs de risque cancer gastrique',
    'Stadification glioblastome',
]

print('='*70)
for q in test_qs:
    print(f'\n❓ {q}')
    results = retriever.search(q, k=3)
    for r in results:
        print(f"   [{r['score']:.3f}] {r['type']:25s} | {r['text'][:120]}...")


❓ Quel est le traitement du cancer de la prostate à un stade avancé ?
   [2.875] treatment_protocol        | [Cancer de la prostate] [traitement_protocole] Protocole: Intermediate-Risk Localized Prostate Cancer — Definitive Local...
   [2.812] treatment_protocol        | [Cancer de la prostate] [traitement_protocole] Protocole: High-Risk and Very High-Risk Localized / Locally Advanced Pros...
   [2.600] treatment_protocol        | [Cancer de la prostate] [traitement_protocole] Protocole: Very Low-Risk and Low-Risk Localized Prostate Cancer — Active ...

❓ Effets secondaires chimiothérapie cancer du poumon petites cellules
   [1.717] drug                      | [Cancer du poumon à petites cellules] [médicament] Médicament: Topotécan. Classe: Inhibiteur de la topoisomérase I. Méca...
   [1.648] drug                      | [Cancer du poumon à petites cellules] [médicament] Médicament: Étoposide. Classe: Inhibiteur de la topoisomérase II. Méc...
   [1.589] drug                      | [Can

## 6. Prompt adaptatif + pipeline RAG

In [17]:
PROMPT_STRUCTURES = {
    'traitement':         'Structure: 1) Lignes de traitement, 2) Médicaments et doses, 3) Critères de choix.',
    'diagnostic':         'Structure: 1) Examens initiaux, 2) Confirmation diagnostique, 3) Staging.',
    'pronostic':          'Structure: 1) Facteurs pronostiques, 2) Survie par stade, 3) Récidive.',
    'effets_secondaires': 'Structure: 1) Effets fréquents, 2) Effets graves (grade 3-4), 3) Prise en charge.',
    'suivi':              'Structure: 1) Calendrier de suivi, 2) Examens recommandés, 3) Signes d\'alarme.',
    'facteurs_risque':    'Structure: 1) Facteurs non modifiables, 2) Facteurs modifiables, 3) Dépistage.',
    'default':            'Réponds de manière structurée et cliniquement pertinente.',
}

def format_context(results):
    return '\n\n'.join(
        f'[Source {i+1} | {r["type"]} | score:{r["score"]:.3f}]\n{r["text"]}'
        for i, r in enumerate(results)
    )

def build_prompt(query, context):
    cancer = detect_cancer(query) or 'oncologie'
    qtype  = detect_qtype(query)
    struct = PROMPT_STRUCTURES.get(qtype, PROMPT_STRUCTURES['default'])
    return f'''<|im_start|>system
Tu es un assistant médical expert en oncologie, spécialisé en {cancer}.
Réponds UNIQUEMENT à partir du contexte fourni. Si l\'information manque, dis-le.
{struct}
Réponds en français. Sois précis et concis.
<|im_end|>
<|im_start|>user
CONTEXTE:\n{context}\n\nQUESTION: {query}
<|im_end|>
<|im_start|>assistant
'''

def rag_answer(query, k=4, verbose=True):
    t0 = time.time()
    cancer = detect_cancer(query)
    qtype  = detect_qtype(query)
    if verbose:
        print(f'🎯 Cancer: {cancer or "aucun"} | Type: {qtype}')
    results = retriever.search(query, k=k, cancer_filter=cancer, strict=bool(cancer), qtype=qtype)
    if not results:
        results = retriever.search(query, k=k, qtype=qtype)
    context = format_context(results)
    prompt  = build_prompt(query, context)
    # answer = llm.generate(prompt, max_new_tokens=400)  # décommenter si LLM chargé
    answer = '[LLM non chargé]'
    if verbose:
        print(f'\n📄 {len(results)} chunks récupérés :')
        for r in results:
            print(f"   [{r['score']:.3f}] {r['type']:25s} {r['text'][:100]}...")
    return {'query': query, 'cancer': cancer, 'qtype': qtype,
            'answer': answer, 'sources': results, 'time': round(time.time()-t0,2)}

print('✅ Pipeline RAG prêt')
result = rag_answer('Quel est le traitement du cancer de la prostate à un stade avancé ?')

✅ Pipeline RAG prêt
🎯 Cancer: Cancer de la prostate | Type: traitement

📄 4 chunks récupérés :
   [4.600] treatment_protocol        [Cancer de la prostate] [traitement_protocole] Protocole: Intermediate-Risk Localized Prostate Cance...
   [4.499] treatment_protocol        [Cancer de la prostate] [traitement_protocole] Protocole: High-Risk and Very High-Risk Localized / L...
   [4.160] treatment_protocol        [Cancer de la prostate] [traitement_protocole] Protocole: Very Low-Risk and Low-Risk Localized Prost...
   [3.956] treatment_protocol        [Cancer de la prostate] [traitement_protocole] Protocole: Metastatic Castration-Resistant Prostate C...


## 7. LLM Phi-3 (optionnel)

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

class Phi3Generator:
    def __init__(self, model_name='microsoft/Phi-3-mini-4k-instruct'):
        self.model_name = model_name
        self.tokenizer = self.model = None

    def load(self):
        print(f'🔄 Chargement {self.model_name}...')
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, trust_remote_code=True,
            torch_dtype=torch.float32, low_cpu_mem_usage=True)
        self.model.eval()
        print('✅ Phi-3 chargé')

    def generate(self, prompt, max_new_tokens=400):
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3000)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                repetition_penalty=1.1, pad_token_id=self.tokenizer.eos_token_id)
        # Retourner seulement la partie générée
        return self.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# Décommentez pour activer :
# llm = Phi3Generator()
# llm.load()
print('✅ Classe Phi3Generator définie — décommentez les 2 lignes pour charger')

✅ Classe Phi3Generator définie — décommentez les 2 lignes pour charger


## 8. Évaluation Hit@3

In [19]:
eval_set = [
    {'q': 'Protocole traitement prostate stade avancé',           'cancer': 'Cancer de la prostate',                         'expected': 'treatment_protocol'},
    {'q': 'Effets secondaires étoposide cancer poumon',           'cancer': 'Cancer du poumon à petites cellules',           'expected': 'toxicity_management'},
    {'q': 'Suivi après traitement cancer thyroïde',               'cancer': 'Cancer de la thyroïde',                         'expected': 'followup'},
    {'q': 'Résistance enzalutamide cancer prostate',              'cancer': 'Cancer de la prostate',                         'expected': 'resistance_mechanism'},
    {'q': 'Stadification TNM cancer sein',                        'cancer': 'Cancer du sein',                                'expected': 'staging_system'},
    {'q': 'Traitement glioblastome sujet âgé',                    'cancer': 'Gliomes de haut grade',                         'expected': 'treatment_protocol'},
    {'q': 'Diagnostic cancer ovaire biomarqueurs',                'cancer': "Cancer épithélial de l'ovaire",                 'expected': 'biomarker_genetics'},
    {'q': 'Métastases cérébrales mélanome traitement',            'cancer': 'Mélanome',                                      'expected': 'metastasis'},
    {'q': 'Contre-indications cisplatine cancer gastrique',       'cancer': 'Cancer gastrique',                              'expected': 'contraindication_interaction'},
    {'q': 'Urgences oncologiques cancer colorectal',              'cancer': 'Cancer Colorectal (CCR)',                       'expected': 'oncology_emergency'},
]

print('='*70)
print('📊 ÉVALUATION Hit@3')
print('='*70)

hits = 0
for item in eval_set:
    results = retriever.search(item['q'], k=3, cancer_filter=item['cancer'], strict=True)
    found   = [r['type'] for r in results]
    hit     = item['expected'] in found
    hits   += int(hit)
    print(f"{'✅' if hit else '❌'} {item['q'][:48]:48s} | attendu: {item['expected']:25s} | trouvé: {found}")

print(f'\n🎯 Score Hit@3 : {hits}/{len(eval_set)} = {100*hits/len(eval_set):.0f}%')

📊 ÉVALUATION Hit@3
✅ Protocole traitement prostate stade avancé       | attendu: treatment_protocol        | trouvé: ['treatment_protocol', 'treatment_protocol', 'treatment_protocol']
❌ Effets secondaires étoposide cancer poumon       | attendu: toxicity_management       | trouvé: ['drug', 'drug', 'drug']
✅ Suivi après traitement cancer thyroïde           | attendu: followup                  | trouvé: ['followup', 'followup', 'followup']
✅ Résistance enzalutamide cancer prostate          | attendu: resistance_mechanism      | trouvé: ['resistance_mechanism', 'drug', 'contraindication_interaction']
✅ Stadification TNM cancer sein                    | attendu: staging_system            | trouvé: ['treatment_protocol', 'staging_system', 'staging_system']
✅ Traitement glioblastome sujet âgé                | attendu: treatment_protocol        | trouvé: ['treatment_protocol', 'clinical_case', 'medical_literature']
✅ Diagnostic cancer ovaire biomarqueurs            | attendu: biomarker_geneti

In [20]:
# ═══════════════════════════════════════════════════════════════════
# NOUVELLES QUESTIONS DE TEST
# ═══════════════════════════════════════════════════════════════════

nouvelles_questions = [
    # ── TRAITEMENT ──────────────────────────────────────────────────
    "Quel est le protocole de première ligne pour le cancer du sein HER2 positif ?",
    "Quelle est la prise en charge du cancer colorectal métastatique ?",
    
    # ── EFFETS SECONDAIRES (pour tester toxicity_management) ────────
    "Quels sont les effets secondaires de l'immunothérapie dans le mélanome ?",
    "Toxicité cardiaque des anthracyclines dans le cancer du sein",
    "Prise en charge des nausées et vomissements post-chimiothérapie",
    
    # ── DIAGNOSTIC ──────────────────────────────────────────────────
    "Quels sont les marqueurs tumoraux pour le cancer colorectal ?",
    "Comment faire le diagnostic du cancer du rein ?",
    
    # ── PRONOSTIC / SURVIE ──────────────────────────────────────────
    "Quel est le pronostic du glioblastome chez le sujet âgé ?",
    "Facteurs pronostiques du cancer de la vessie",
    
    # ── URGENCES ONCOLOGIQUES (pour oncology_emergency) ─────────────
    "Que faire en cas de compression médullaire chez un patient cancéreux ?",
    "Prise en charge d'une hypercalcémie maligne",
    
    # ── MÉTASTASES ──────────────────────────────────────────────────
    "Où se font les métastases du cancer du rein ?",
    "Traitement des métastases hépatiques du cancer colorectal",
    
    # ── RÉSISTANCE ──────────────────────────────────────────────────
    "Mécanismes de résistance au trastuzumab dans le cancer du sein",
    
    # ── SUIVI ───────────────────────────────────────────────────────
    "Quel suivi après un cancer du col de l'utérus ?",
]

# Exécution des tests
print("=" * 70)
print("🧪 TESTS DU RETRIEVER AMÉLIORÉ - NOUVELLES QUESTIONS")
print("=" * 70)

for q in nouvelles_questions:
    print(f"\n❓ {q}")
    results = retriever.search_auto(q, k=3)
    for r in results:
        print(f"   [{r['score']:.3f}] {r['type']:25s} | {r['text'][:100]}...")
    print("-" * 70)

🧪 TESTS DU RETRIEVER AMÉLIORÉ - NOUVELLES QUESTIONS

❓ Quel est le protocole de première ligne pour le cancer du sein HER2 positif ?
   [4.581] treatment_protocol        | [Cancer du sein] [traitement_protocole] Protocole: Cancer du sein HER2+ - Néo-adjuvant et Adjuvant (...
   [4.213] treatment_protocol        | [Cancer du sein] [traitement_protocole] Protocole: Cancer du sein HER2+ Métastatique - 1ère et 2ème ...
   [3.867] treatment_protocol        | [Cancer du sein] [traitement_protocole] Protocole: Cancer du sein RE+/HER2- - Hormonothérapie Adjuva...
----------------------------------------------------------------------

❓ Quelle est la prise en charge du cancer colorectal métastatique ?
   [1.270] staging_system            | [Cancer Colorectal (CCR)] [stadification] Stadification — Cancer: Cancer Colorectal (CCR). Système: ...
   [1.234] cancer_knowledge          | [Cancer Colorectal (CCR)] [connaissance_epidemiology] Épidémiologie: global_incidence: Le cancer col...
   [1.192] c